# Notebook 2: Prompt Engineering Aplicado a AWS Bedrock

En este notebook aprenderás a:
- Aplicar técnicas de prompting: zero-shot, few-shot y chain-of-thought
- Usar system prompts para controlar el comportamiento del modelo
- Comparar outputs de distintos modelos para el mismo prompt
- Ajustar parámetros de inferencia y medir su impacto

---

> **Requisito:** Haber completado el Notebook 1 o tener boto3 instalado y credenciales AWS configuradas.

## ¿Por qué existe este notebook?

Ya sabes conectarte a un modelo y enviarle un mensaje. Pero la calidad de la respuesta depende casi en su totalidad de **cómo formulas la pregunta**.

Un modelo de IA es como un empleado extraordinariamente capaz pero brutalmente literal. Si le pides algo vago, te da algo vago. Si le das contexto, formato y ejemplos, te da exactamente lo que necesitas.

Eso es el **prompt engineering**: la disciplina de escribir instrucciones que extraigan el máximo del modelo. No es magia, es técnica. Y hay patrones concretos que funcionan.

Este notebook te enseña los cuatro principales.

## 1. Setup inicial

Igual que en el Notebook 1: cargar credenciales, crear cliente y definir la función auxiliar `invocar_claude`. Aquí la función tiene una novedad: el parámetro `system`, que veremos en la sección 5.

In [ ]:
# !pip install boto3 --quiet

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
AWS_BEARER_TOKEN_BEDROCK=os.getenv("AWS_BEARER_TOKEN_BEDROCK")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION=os.getenv("AWS_DEFAULT_REGION")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

### La función `invocar_claude` con soporte de system prompt

Esta versión de `invocar_claude` añade el parámetro opcional `system`. Si se proporciona, se añade al body antes de los mensajes del usuario.

El **system prompt** es un texto especial que le dice al modelo quién es, cómo debe comportarse y qué restricciones tiene. No es un mensaje más del usuario: es la "personalidad" del modelo para esa sesión.

In [ ]:
import boto3
import json
import time

REGION   = AWS_DEFAULT_REGION
MODEL_ID = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

def invocar_claude(prompt, system=None, temperature=0.7, max_tokens=1024):
    """Helper genérico para invocar Claude en Bedrock."""
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        body["system"] = system

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    return resultado["content"][0]["text"]

print("✅ Setup listo")

## 2. Zero-Shot Prompting

### ¿Qué es?

"Zero-shot" significa **sin ejemplos**. Le das al modelo una instrucción directa y confías en que entiende la tarea por sí solo.

Es el punto de partida más simple. Funciona bien para tareas generales y directas. Cuando falla (formato incorrecto, respuesta demasiado vaga), es cuando escalas a las técnicas siguientes.

### Por qué `temperature=0.0` aquí

En clasificación queremos consistencia, no creatividad. Con temperature 0.0 el modelo siempre elige la opción más probable, lo que hace las respuestas repetibles y predecibles.

**Entra:** un texto a clasificar  
**Sale:** una de tres palabras: POSITIVO, NEGATIVO o NEUTRO

In [ ]:
# Clasificación de sentimiento en modo zero-shot
prompt_zero_shot = """
Clasifica el sentimiento del siguiente texto como POSITIVO, NEGATIVO o NEUTRO.
Responde únicamente con una de esas tres palabras.

Texto: "El servicio fue lento pero la comida estaba deliciosa."
"""

respuesta = invocar_claude(prompt_zero_shot, temperature=0.0)
print(f"Zero-Shot → {respuesta}")

### Clasificar varios textos en bucle

Aquí se aplica el mismo prompt a cuatro textos distintos. La clave es que el prompt es exactamente igual para todos: solo cambia el texto a clasificar.

**Entra:** lista de textos  
**Sale:** etiqueta de sentimiento para cada uno

In [ ]:
# Probar con varios textos
textos = [
    "Me encantó la película, la recomendaría a todos.",
    "El producto llegó roto y tardó dos semanas.",
    "El paquete llegó hoy por la tarde.",
    "Increíble experiencia, superó todas mis expectativas."
]

print("Clasificación de sentimientos (Zero-Shot)")
print("-" * 50)
for texto in textos:
    prompt = f"""Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRO.
Responde solo con la etiqueta.

Texto: \"{texto}\""""
    sentimiento = invocar_claude(prompt, temperature=0.0)
    print(f"[{sentimiento.strip():8}] {texto}")

## 3. Few-Shot Prompting

### ¿Qué es?

"Few-shot" significa **con ejemplos**. En lugar de solo explicarle la tarea al modelo, le muestras ejemplos de entrada y salida correctos.

### ¿Cuándo usarlo en lugar de zero-shot?

Cuando necesitas:
- Un **formato de salida específico** (JSON, tabla, XML)
- Un **estilo concreto** que no es el comportamiento por defecto del modelo
- Que el modelo **imite** un patrón que no describirías fácilmente con palabras

En este caso, queremos que el modelo devuelva JSON con campos específicos. Mostramos tres ejemplos del formato exacto, y luego le pedimos que haga lo mismo con un texto nuevo.

**Entra:** texto de reseña de producto  
**Sale:** JSON con tres campos: `producto`, `sentimiento` y `aspecto`

In [ ]:
prompt_few_shot = """
Eres un extractor de información de reseñas de productos.
Dado un texto, extrae: producto, sentimiento (POS/NEG/NEU) y aspecto principal.
Responde en formato JSON.

### Ejemplos ###

Texto: "Las zapatillas son cómodísimas, perfectas para correr."
Respuesta: {"producto": "zapatillas", "sentimiento": "POS", "aspecto": "comodidad"}

Texto: "El laptop se calienta mucho después de una hora de uso."
Respuesta: {"producto": "laptop", "sentimiento": "NEG", "aspecto": "temperatura"}

Texto: "El auricular tiene 20 horas de batería según el manual."
Respuesta: {"producto": "auricular", "sentimiento": "NEU", "aspecto": "batería"}

### Tu turno ###

Texto: "La cafetera hace un ruido horrible pero el café sale excelente."
Respuesta:
"""

respuesta_fs = invocar_claude(prompt_few_shot, temperature=0.0)
print("Few-Shot → ", respuesta_fs)

### Verificar que la respuesta es JSON válido

Cuando pedimos al modelo que devuelva JSON, no siempre lo hace perfectamente. Este bloque intenta parsear la respuesta y avisa si algo salió mal.

En un sistema en producción, esta validación es imprescindible antes de usar los datos devueltos por el modelo.

**Entra:** texto de la respuesta del modelo  
**Sale:** diccionario Python con los campos, o aviso de error si el JSON no es válido

In [ ]:
# Verificar que la respuesta es JSON válido
try:
    datos = json.loads(respuesta_fs.strip())
    print("✅ JSON válido:")
    for k, v in datos.items():
        print(f"  {k}: {v}")
except json.JSONDecodeError:
    print("⚠️  La respuesta no es JSON puro. Respuesta:", respuesta_fs)

## 4. Chain-of-Thought (CoT) Prompting

### ¿Qué es?

"Chain of Thought" (cadena de pensamiento) consiste en **pedir al modelo que razone paso a paso** antes de dar la respuesta final.

Los modelos de lenguaje, al generar cada palabra mirando solo lo que ha escrito antes, cometen errores en problemas que requieren varios pasos intermedios. Si los forzamos a escribir esos pasos, el proceso de generación mejora significativamente.

### ¿Cuándo usarlo?

- Problemas matemáticos o lógicos
- Razonamiento con múltiples condiciones
- Análisis donde el "por qué" importa tanto como el "qué"

En este bloque se compara la respuesta con y sin CoT para el mismo problema aritmético.

**Entra:** un enunciado de problema  
**Sale (sin CoT):** solo el número final  
**Sale (con CoT):** el razonamiento completo + el resultado

In [ ]:
problema = """
Una tienda tiene 150 camisetas. Vende el 40% el lunes,
luego recibe un nuevo envío de 60 unidades el martes,
y el miércoles vende 35 unidades más.
¿Cuántas camisetas quedan al final del miércoles?
"""

# Sin Chain-of-Thought
print("=== Sin CoT ===")
prompt_sin_cot = f"{problema}\nResponde solo con el número final."
print(invocar_claude(prompt_sin_cot, temperature=0.0))

print()

# Con Chain-of-Thought
print("=== Con CoT ===")
prompt_con_cot = f"""{problema}
Piensa paso a paso antes de responder:
1. Calcula cuántas se vendieron el lunes
2. Suma el nuevo envío
3. Resta las ventas del miércoles
4. Da el resultado final
"""
print(invocar_claude(prompt_con_cot, temperature=0.0))

## 5. System Prompts: Controlar el rol y comportamiento

### ¿Qué es un system prompt?

Es una instrucción que se envía al modelo **antes** de cualquier mensaje del usuario. Define el contexto, el rol y las restricciones del modelo para toda la conversación.

Piénsalo como el "briefing" que le darías a un empleado antes de atender una llamada: *"Eres el soporte técnico de X, no hablas de precios, usa lenguaje formal"*.

El mismo modelo, con system prompts distintos, se comporta de formas completamente diferentes. En este bloque se hace la misma pregunta a tres "versiones" distintas del modelo.

**Entra:** la misma pregunta (`¿Qué es una API REST?`)  
**Sale:** tres respuestas radicalmente distintas según el rol asignado

In [ ]:
pregunta = "¿Qué es una API REST?"

sistemas = {
    "Experto técnico": "Eres un arquitecto de software senior. Responde de forma técnica y precisa, usando terminología específica del sector.",
    "Profesor para niños": "Eres un profesor de primaria. Explica los conceptos usando analogías simples y ejemplos de la vida cotidiana. Usa un lenguaje muy sencillo.",
    "Vendedor de producto": "Eres un comercial de soluciones tecnológicas. Destaca los beneficios de negocio y el valor que aporta la tecnología al cliente."
}

for rol, system_prompt in sistemas.items():
    print(f"\n{'='*60}")
    print(f"ROL: {rol}")
    print(f"{'='*60}")
    respuesta = invocar_claude(pregunta, system=system_prompt, temperature=0.5, max_tokens=300)
    print(respuesta)

## 6. Comparar modelos con el mismo prompt

### ¿Por qué comparar modelos?

Bedrock da acceso a modelos de distintos proveedores. No todos responden igual de bien a todos los tipos de tarea. Antes de elegir el modelo para tu aplicación, tiene sentido compararlos con tu prompt real.

### El helper `invocar_modelo_generico`

El reto es que cada proveedor tiene su propio formato de body (visto en el Notebook 1). Esta función encapsula esas diferencias: detecta el proveedor por el `model_id` y formatea el body correctamente.

| Si el model_id contiene... | Formato que usa |
|---------------------------|-----------------|
| `"anthropic"` | Messages API de Anthropic |
| `"amazon.titan"` | formato Titan con `inputText` |
| `"meta.llama"` | formato Llama con `prompt` |

**Entra:** nombre y ID de cada modelo + el prompt de comparación  
**Sale:** la respuesta de cada modelo para el mismo prompt

In [ ]:
def invocar_modelo_generico(model_id, prompt, max_tokens=512):
    """Invoca distintos modelos adaptando el formato del body."""
    if "anthropic" in model_id:
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}]
        }
        response = bedrock_runtime.invoke_model(
            modelId=model_id, body=json.dumps(body),
            contentType="application/json", accept="application/json"
        )
        resultado = json.loads(response["body"].read())
        return resultado["content"][0]["text"]

    elif "amazon.titan" in model_id:
        body = {
            "inputText": prompt,
            "textGenerationConfig": {"maxTokenCount": max_tokens, "temperature": 0.7}
        }
        response = bedrock_runtime.invoke_model(
            modelId=model_id, body=json.dumps(body),
            contentType="application/json", accept="application/json"
        )
        resultado = json.loads(response["body"].read())
        return resultado["results"][0]["outputText"]

    elif "meta.llama" in model_id:
        body = {
            "prompt": prompt,
            "max_gen_len": max_tokens,
            "temperature": 0.7
        }
        response = bedrock_runtime.invoke_model(
            modelId=model_id, body=json.dumps(body),
            contentType="application/json", accept="application/json"
        )
        resultado = json.loads(response["body"].read())
        return resultado["generation"]

    return "Modelo no soportado en este helper"


# Modelos a comparar (ajusta según los que tengas habilitados)
modelos_comparar = [
    ("Claude 3 Haiku",  "eu.anthropic.claude-haiku-4-5-20251001-v1:0"),
    ("Llama 3.2 3B Instruct",    "eu.meta.llama3-2-3b-instruct-v1:0"),
]

prompt_comparacion = "Explica el concepto de 'overfitting' en machine learning en máximo 3 oraciones."

print(f"PROMPT: {prompt_comparacion}")
print("=" * 70)

for nombre, model_id in modelos_comparar:
    print(f"\n🤖 {nombre} ({model_id}):")
    print("-" * 50)
    try:
        respuesta = invocar_modelo_generico(model_id, prompt_comparacion)
        print(respuesta.strip())
    except Exception as e:
        print(f"  ⚠️  Error: {e}")
    time.sleep(1)  # Evitar throttling

## 7. Impacto de los parámetros de inferencia

### `temperature`: el dial de creatividad

Ya lo vimos en el Notebook 1, pero aquí lo visualizamos con más claridad: tres valores de temperature para el mismo prompt creativo, para que veas la diferencia real.

**Entra:** el mismo prompt de escritura creativa con tres temperaturas distintas  
**Sale:** tres respuestas distintas, de la más predecible a la más libre

In [ ]:
prompt_creativo = "Escribe el primer párrafo de una historia de ciencia ficción."

configs = [
    {"temperature": 0.0, "label": "🧊 T=0.0 (determinista)"},
    {"temperature": 0.5, "label": "🌡️  T=0.5 (equilibrado)"},
    {"temperature": 1.0, "label": "🔥 T=1.0 (muy creativo)"},
]

for config in configs:
    print(f"\n{config['label']}")
    print("-" * 40)
    respuesta = invocar_claude(prompt_creativo, temperature=config["temperature"], max_tokens=200)
    print(respuesta.strip())
    time.sleep(0.5)

### Verificar el determinismo con temperature=0

Con `temperature=0.0`, el modelo debería dar **exactamente la misma respuesta** en cada intento. Este bloque lo verifica haciendo tres llamadas idénticas.

Esto es importante en aplicaciones donde la consistencia es crítica: si llamas dos veces con el mismo input, quieres el mismo output.

In [ ]:
# Demostrar la determinismo de temperature=0
print("Verificando determinismo con temperature=0.0:")
prompt_corto = "Di exactamente 5 palabras sobre el espacio."

for i in range(3):
    r = invocar_claude(prompt_corto, temperature=0.0, max_tokens=50)
    print(f"  Intento {i+1}: {r.strip()}")
    time.sleep(0.5)

## 8. Prompt Templates reutilizables

### ¿Por qué usar templates?

En cualquier aplicación real, el prompt no es fijo: cambia según el usuario, el documento, el idioma, etc. Hardcodear el prompt con f-strings por todas partes es un desastre de mantenimiento.

La solución: crear una clase `PromptTemplate` que separa la **estructura del prompt** (fija) de los **valores que cambian** (variables).

### Cómo funciona

La clase recibe un string con `{placeholders}` y el método `.format()` los rellena con los valores que le pasas.

**Entra:** número de puntos, tono y texto a resumir  
**Sale:** un prompt completo y formateado, listo para enviarse al modelo

In [ ]:
class PromptTemplate:
    """Clase simple para gestionar templates de prompts."""

    def __init__(self, template: str):
        self.template = template

    def format(self, **kwargs) -> str:
        return self.template.format(**kwargs)


# Template de resumen
template_resumen = PromptTemplate("""
Resume el siguiente texto en {num_puntos} puntos clave.
Usa un tono {tono}.
Formato: lista numerada.

TEXTO:
{texto}

RESUMEN:
""")

texto_ejemplo = """
AWS Bedrock es un servicio completamente gestionado que ofrece una selección de modelos
de fundación de alto rendimiento de las principales empresas de IA como Anthropic,
Cohere, Meta, Mistral y Amazon, a través de una única API. Bedrock permite construir
aplicaciones de IA generativa de forma segura, privada y responsable sin necesidad de
gestionar infraestructura. Los datos de los clientes nunca se usan para entrenar los
modelos base y se pueden aplicar controles de seguridad a nivel empresarial.
"""

prompt_final = template_resumen.format(
    num_puntos=3,
    tono="formal y conciso",
    texto=texto_ejemplo
)

print("=== Prompt generado ===")
print(prompt_final)
print("\n=== Respuesta ===")
print(invocar_claude(prompt_final, temperature=0.3))

## 9. Buenas prácticas de Prompt Engineering

Un resumen de cuándo usar cada técnica:

| Técnica | Cuándo usarla | Ejemplo |
|--------|--------------|--------|
| **Zero-shot** | Tareas simples y directas | Clasificar, traducir, resumir |
| **Few-shot** | Formatos de salida específicos | JSON estructurado, tablas |
| **Chain-of-Thought** | Razonamiento complejo | Matemáticas, lógica, análisis |
| **System Prompt** | Definir rol y restricciones | Chatbots especializados |
| **Temperature baja** | Respuestas consistentes | Datos, código, clasificación |
| **Temperature alta** | Creatividad y variedad | Brainstorming, escritura creativa |

---

**Siguiente notebook →** Embeddings y Búsqueda Semántica con Bedrock